In [ ]:
# Ref: https://developers.llamaindex.ai/python/framework/integrations/vector_stores/elasticsearchindexdemo/

In [1]:
%pip install -qU llama-index-vector-stores-elasticsearch 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
# Ollama related
# https://docs.llamaindex.ai/en/stable/examples/embeddings/ollama_embedding/
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama
from llama_index.core import SimpleDirectoryReader

In [15]:
from llama_index.core import Settings

llm = Ollama(
                 model=  "deepseek-r1:7b", #  works best for router agent, # "qwen3.5:4b", #      "llama3.2:latest",
                 request_timeout=3600,  # Change it
                 #temperature = 0.4,
                 context_window=4096,    # Limit timeouts
                 #mirostat = 0
            )

Settings.llm = llm

embed_model = OllamaEmbedding(
                              model_name="nomic-embed-text",      # Using foundational model may be overkill
                              base_url="http://localhost:11434",
                             )

Settings.embed_model = embed_model

In [16]:
from llama_index.vector_stores.elasticsearch import ElasticsearchStore

es = ElasticsearchStore(
    index_name="my_index",
    es_url="http://localhost:9200",
)

In [17]:
from llama_index.core.schema import TextNode

movies = [
    TextNode(
        text="The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption.",
        metadata={"title": "Pulp Fiction"},
    ),
    TextNode(
        text="When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.",
        metadata={"title": "The Dark Knight"},
    ),
    TextNode(
        text="An insomniac office worker and a devil-may-care soapmaker form an underground fight club that evolves into something much, much more.",
        metadata={"title": "Fight Club"},
    ),
    TextNode(
        text="A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.",
        metadata={"title": "Inception"},
    ),
    TextNode(
        text="A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its controllers.",
        metadata={"title": "The Matrix"},
    ),
    TextNode(
        text="Two detectives, a rookie and a veteran, hunt a serial killer who uses the seven deadly sins as his motives.",
        metadata={"title": "Se7en"},
    ),
    TextNode(
        text="An organized crime dynasty's aging patriarch transfers control of his clandestine empire to his reluctant son.",
        metadata={"title": "The Godfather", "theme": "Mafia"},
    ),
]

In [18]:
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.elasticsearch import ElasticsearchStore

In [19]:
def print_results(results):
    for rank, result in enumerate(results, 1):
        print(
            f"{rank}. title={result.metadata['title']} score={result.get_score()} text={result.get_text()}"
        )


In [20]:
def search(
    vector_store: ElasticsearchStore, nodes: list[TextNode], query: str
):
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex(nodes, storage_context=storage_context)

    print(">>> Documents:")
    retriever = index.as_retriever()
    results = retriever.retrieve(query)
    print_results(results)

    print("\n>>> Answer:")
    query_engine = index.as_query_engine()
    response = query_engine.query(query)
    print(response)

In [21]:
from llama_index.vector_stores.elasticsearch import AsyncDenseVectorStrategy

dense_vector_store = ElasticsearchStore(
    es_url="http://localhost:9200",  # for Elastic Cloud authentication see above
    index_name="movies_dense",
    retrieval_strategy=AsyncDenseVectorStrategy(),
)

search(dense_vector_store, movies, "which movie involves dreaming?")

>>> Documents:
1. title=Inception score=1.0 text=A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.
2. title=Inception score=1.0 text=A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.

>>> Answer:
The movie that involves the concept of dreaming is **Inception**.


In [22]:
search(dense_vector_store, movies, "which movie involves dreaming?")

>>> Documents:
1. title=Inception score=1.0 text=A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.
2. title=Inception score=1.0 text=A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into thed of a C.E.O.

>>> Answer:
The movie involving the concept of dreaming is **Inception**.


**::Q::**   
> In the example for the ElasticSearch vector store, I have several TextNodes taking the inputs. TextNode takes two inputs, text and metadata. Can I have a code something like this: movies=[ TextNode( text = SomeFileReader(Textfilename1), metadata={"title" : "case number xyz1"} ), TextNode( text = SomeFileReader(Textfilename2), metadata={"title" : "case number xyz2"} ), ] If yes, please suggest a script where I read multiple files, and for each I add metadata separately. Also can TextNode take .md files as input

**Ans**:
>Yes, you can create a list of TextNodes by reading multiple files and assigning separate metadata to each, as shown in your example. You simply need to read the file content (regardless of extension, including .md) and pass it as the text argument to TextNode, with metadata as a dictionary. TextNode does not care about file type; it just takes a string for text, so .md files are fine if you read them as text.

In [ ]:
from llama_index.core.schema import TextNode

filenames = ["file1.md", "file2.md"]
movies = []
for fname in filenames:
    with open(fname, "r", encoding="utf-8") as f:
        content = f.read()
    movies.append(TextNode(text=content, metadata=("title": f"case number {fname)"}))


**::Q::**
>Thanks. Please change the above example code where all files are read automatically from a folder without requiring their prior listing .

In [ ]:
import os
from llama_index.core.schema import TextNode

folder = "./your_folder"
movies = []
for fname in os.listdir(folder):
    path = os.path.join(folder, fname)
    if os.path.isfile(path):
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        movies.append(TextNode(text=content, metadata=("title": f"case number {fname)"}))


**::Q::**
>Ok. Let me put it differently. In the above example, a file is read from a folder and its title is taken as the metadata for the TextNode. Now all my files are several sub-folders (and not directly in the parent folder). Each sub-folder has a meaningful name. My code should read all text files from each one of the sub-folders, one-by-one, and now text of each file read has two metadata points: one is the filename as 'title' and the other is the sub-folder name as 'region'. What would be code that creates several TextNodes from each file read and appends them to movies = []

In [ ]:
import os
from llama_index.core.schema import TextNode

parent_folder = "./your_parent_folder"
movies = []
for root, dirs, files in os.walk(parent_folder):
    subfolder = os.path.basename(root)
    for fname in files:
        path = os.path.join(root, fname)
        if os.path.isfile(path):
            with open(path, "r", encoding="utf-8") as f:
                content = f.read()
            movies.append(TextNode(
                text=content,
                metadata=("title": fname, "region": subfolder)
            ))


>This will create a TextNode for every file in every subfolder, with title as the filename and region as the subfolder name,

**::Q::**
Can TextNode be also used when the storage_context is PGVector and not elasticsearch. If yes what would be the code?

>Yes, you can use TextNode with PGVectorStore as your vector store backend. TextNode is a general-purpose node type in LlamaIndex and works with any supported vector store, including PGVector. The process is the same: create TextNodes, then build a VectorStoreIndex with a StorageContext that uses PGVectorStore.

In [ ]:
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.schema import TextNode

# Assume movies is a list of TextNode objects as before
vector_store = PGVectorStore.from_params(
    database="vector_db",
    host="localhost",
    password="password",
    port=5432,
    user="postgres",
    table_name="my_table",
    embed_dim=1536  # embedding dimension
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex(movies, storage_context=storage_context)
